# Agents as Tools with Strands Agents


"Agents as Tools" is an architectural pattern in AI systems where specialized AI agents are wrapped as callable functions (tools) that can be used by other agents. This creates a hierarchical structure where:

1. A primary "orchestrator" agent handles user interaction and determines which specialized agent to call

2. Specialized "tool agents" perform domain-specific tasks when called by the orchestrator

This approach mimics human team dynamics, where a manager coordinates specialists, each bringing unique expertise to solve complex problems. Rather than a single agent trying to handle everything, tasks are delegated to the most appropriate specialized agent.



## Key Benefits and Core Principles

The "Agents as Tools" pattern offers several advantages:

- Separation of concerns: Each agent has a focused area of responsibility, making the system easier to understand and maintain
- Hierarchical delegation: The orchestrator decides which specialist to invoke, creating a clear chain of command
- Modular architecture: Specialists can be added, removed, or modified independently without affecting the entire system
- Improved performance: Each agent can have tailored system prompts and tools optimized for its specific task


In [1]:
!pip install -r requirements.txt

In [2]:
import os

from strands import Agent, tool
from strands_tools import file_write


In this module we will be creating an orchestrator based multi-agent workflow. 

<div style="text-align:left">
    <img src="images/architecture.png" width="75%" />
</div>

We will also explore `use_llm` which allows use to create nested agents.

## Research Agent

Lets first create a basic reasearch assistant with http_request tool. 

In [3]:
RESEARCH_ASSISTANT_PROMPT = """You are a specialized research assistant. Focus only on providing
factual, well-sourced information in response to research questions.
Always cite your sources when possible."""

In [4]:
research_agent = Agent(
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt=RESEARCH_ASSISTANT_PROMPT,
    # tools=[http_request]  # Here you can enable an agentic ai search tool
)

query = "Overview of Amazon Bedrock and its features"
# Call the agent and return its response
response = research_agent(query)

# Amazon Bedrock Overview

Amazon Bedrock is a fully managed service launched by AWS in 2023 that provides access to foundation models (FMs) from leading AI companies through a single API.

## Key Features

### 1. **Multiple Foundation Models**
- Access to models from **Anthropic (Claude)**, **AI21 Labs**, **Cohere**, **Meta (Llama)**, **Stability AI**, and **Amazon Titan**
- Ability to choose models based on specific use case requirements
- Support for text, image, and embedding generation

### 2. **Model Customization**
- **Fine-tuning**: Customize models with your own labeled datasets
- **Continued pre-training**: Adapt models to specific domains with unlabeled data
- Private customization - your data remains secure and isn't used to train base models

### 3. **Retrieval-Augmented Generation (RAG)**
- **Knowledge Bases**: Connect FMs to your proprietary data sources
- Integrates with vector databases for contextual information retrieval
- Improves accuracy and reduces hallucinations

Now we can wrap this agent as a tool. Allowing other agents to interact with it. 

####  Best Practices for Agent as Tools

When implementing the "Agents as Tools" pattern with Strands Agents:

1. Clear tool documentation: Write descriptive docstrings that explain the agent's expertise
2. Focused system prompts: Keep each specialized agent tightly focused on its domain
3. Proper response handling: Use consistent patterns to extract and format responses
4. Tool selection guidance: Give the orchestrator clear criteria for when to use each specialized agen

In [5]:
@tool
def research_assistant(query: str) -> str:
    """
    Process and respond to research-related queries.

    Args:
        query: A research question requiring factual information

    Returns:
        A detailed research answer with citations
    """
    try:
        # Strands agents makes it easy to create a specialized agent
        research_agent = Agent(
            model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
            system_prompt=RESEARCH_ASSISTANT_PROMPT,
        )

        # Call the agent and return its response
        response = research_agent(query)
        return str(response)
    except Exception as e:
        return f"Error in research assistant: {str(e)}"

Now lets follow the best practices and create `product_recommendation_assistant`, `trip_planning_assistant`, and `orchestrator` agent.

### Product Recommendation Assistant

In [6]:
@tool
def product_recommendation_assistant(query: str) -> str:
    """
    Handle product recommendation queries by suggesting appropriate products.

    Args:
        query: A product inquiry with user preferences

    Returns:
        Personalized product recommendations with reasoning
    """
    try:
        product_agent = Agent(
            model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",  # Optional: Specify the model ID
            system_prompt="""You are a specialized product recommendation assistant.
            Provide personalized product suggestions based on user preferences. Always cite your sources.""",
        )
        # Call the agent and return its response
        response = product_agent(query)

        return str(response)
    except Exception as e:
        return f"Error in product recommendation: {str(e)}"

In [7]:
product_recommendation_assistant("Product recommendations for flying cars")

# Flying Car Product Recommendations

I need to clarify that **fully autonomous flying cars for consumer purchase are not yet commercially available** as of my last update. However, here are the closest alternatives and emerging options:

## 1. **Currently Available/Pre-Order**

### **PAL-V Liberty** (Netherlands)
- **Type**: Gyroplane/car hybrid
- **Price**: ~$400,000 - $600,000
- **Status**: Limited production, requires pilot license
- **Features**: Road-legal car that converts to gyrocopter; 2-seater

### **Samson Sky Switchblade** (USA)
- **Price**: ~$170,000 (reservation holders)
- **Status**: Pre-orders accepted, production pending
- **Features**: 3-wheel vehicle with foldable wings

## 2. **Electric VTOL (Near Future)**

### **Joby Aviation eVTOL**
- **Expected**: 2025+ (air taxi service first)
- **Type**: Electric vertical takeoff aircraft
- **Note**: Initially commercial service, not personal ownership

### **Lilium Jet**
- **Type**: Electric jet-powered VTOL
- **Status**: In 

"# Flying Car Product Recommendations\n\nI need to clarify that **fully autonomous flying cars for consumer purchase are not yet commercially available** as of my last update. However, here are the closest alternatives and emerging options:\n\n## 1. **Currently Available/Pre-Order**\n\n### **PAL-V Liberty** (Netherlands)\n- **Type**: Gyroplane/car hybrid\n- **Price**: ~$400,000 - $600,000\n- **Status**: Limited production, requires pilot license\n- **Features**: Road-legal car that converts to gyrocopter; 2-seater\n\n### **Samson Sky Switchblade** (USA)\n- **Price**: ~$170,000 (reservation holders)\n- **Status**: Pre-orders accepted, production pending\n- **Features**: 3-wheel vehicle with foldable wings\n\n## 2. **Electric VTOL (Near Future)**\n\n### **Joby Aviation eVTOL**\n- **Expected**: 2025+ (air taxi service first)\n- **Type**: Electric vertical takeoff aircraft\n- **Note**: Initially commercial service, not personal ownership\n\n### **Lilium Jet**\n- **Type**: Electric jet-powe

### Trip Planning Assistant

In [8]:
@tool
def trip_planning_assistant(query: str) -> str:
    """
    Create travel itineraries and provide travel advice.

    Args:
        query: A travel planning request with destination and preferences

    Returns:
        A detailed travel itinerary or travel advice
    """
    try:
        travel_agent = Agent(
            model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",  # Optional: Specify the model ID
            system_prompt="""You are a specialized travel planning assistant.
            Create detailed travel itineraries based on user preferences.""",
        )
        # Call the agent and return its response
        response = travel_agent(query)

        return str(response)
    except Exception as e:
        return f"Error in trip planning: {str(e)}"

### Orchestrator Agent

In [9]:
# Define orchestrator system prompt with clear tool selection guidance
MAIN_SYSTEM_PROMPT = """
You are an assistant that routes queries to specialized agents:
- For research questions and factual information → Use the research_assistant tool
- For product recommendations and shopping advice → Use the product_recommendation_assistant tool
- For travel planning and itineraries → Use the trip_planning_assistant tool
- For simple questions not requiring specialized knowledge → Answer directly

Always select the most appropriate tool based on the user's query.
"""

In [10]:
# Strands Agents allows easy integration of agent tools
orchestrator = Agent(
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",  # Optional: Specify the model ID
    system_prompt=MAIN_SYSTEM_PROMPT,
    tools=[
        research_assistant,
        product_recommendation_assistant,
        trip_planning_assistant,
        file_write,
    ],
)

In [18]:
# Example: E-commerce Customer Service System
customer_query = (
    "I'm looking for hiking boots. Write the final response to current directory."
)

os.environ["BYPASS_TOOL_CONSENT"] = "true"

# The orchestrator automatically determines this requires multiple specialized agents
response = orchestrator(customer_query)


Tool #4: product_recommendation_assistant
I'd be happy to help you find hiking boots! To provide the best recommendations, I need to know a bit more about your needs:

1. **What type of hiking do you plan to do?**
   - Day hikes on maintained trails
   - Multi-day backpacking trips
   - Technical mountain terrain
   - Light trail walking

2. **What's your budget range?**
   - Under $100
   - $100-$200
   - $200-$300
   - Over $300

3. **Do you have any specific preferences?**
   - Waterproofing needs
   - Ankle support level (low, mid, or high cut)
   - Weight preference (lightweight vs. durability)
   - Wide or narrow feet
   - Any foot conditions or concerns

4. **What climate/terrain will you be hiking in?**
   - Wet/rainy conditions
   - Dry/desert terrain
   - Rocky mountains
   - Muddy trails

Once you share these details, I can recommend specific boots that match your needs!
Tool #5: file_write


╔═════ File Write Operation ═════╗
║                                ║
║ Path: hiking_boots_inquiry.txt ║
║ Size: 955 characters           ║
║                                ║
╚════════════════════════════════╝

╔══════════════════ Write Successful ═══════════════════╗
║ File written successfully to hiking_boots_inquiry.txt ║
╚═══════════════════════════════════════════════════════╝

I've written the hiking boots inquiry response to **hiking_boots_inquiry.txt** in your current directory. 

To help you find the perfect hiking boots, I'll need some additional information about:
- The type of hiking you'll be doing
- Your budget range
- Specific preferences (waterproofing, ankle support, etc.)
- The climate and terrain conditions

Once you provide these details, I can give you specific boot recommendations tailored to your needs!

In [32]:
# Example: E-commerce Customer Service System
customer_query = (
    "I'm looking for best shoes to walk in patagonia Write the final response to current directory."
)

os.environ["BYPASS_TOOL_CONSENT"] = "true"

# The orchestrator automatically determines this requires multiple specialized agents
response = orchestrator(customer_query)


Tool #14: product_recommendation_assistant
# Best Hiking Footwear for Patagonia

For Patagonia's challenging conditions (rocky terrain, river crossings, unpredictable weather, and multi-day treks), here are my top recommendations:

## **Top Hiking Boots**

### 1. **Salomon Quest 4 GTX**
- **Why**: Excellent ankle support, Gore-Tex waterproofing, superior grip on wet rocks
- **Best for**: Multi-day treks with heavy packs
- **Features**: Contagrip outsole, 4D chassis stability, durable construction
- **Price range**: $230-260

### 2. **La Sportiva Nucleo High II GTX**
- **Why**: Outstanding durability for rugged Patagonian terrain
- **Best for**: Technical trails and scree
- **Features**: Vibram Nano-Tech outsole, reinforced toe/heel, excellent waterproofing
- **Price range**: $270-300

### 3. **Scarpa Zodiac Plus GTX**
- **Why**: Mountain boot performance with trekking comfort
- **Best for**: Variable terrain including snow patches
- **Features**: Crampon-compatible, Vibram Pentax Prec

╔══════════════ File Write Operation ══════════════╗
║                                                  ║
║ Path: patagonia_hiking_shoes_recommendations.txt ║
║ Size: 5874 characters                            ║
║                                                  ║
╚══════════════════════════════════════════════════╝

╔═══════════════════════════ Write Successful ════════════════════════════╗
║ File written successfully to patagonia_hiking_shoes_recommendations.txt ║
╚═════════════════════════════════════════════════════════════════════════╝

Perfect! I've created a comprehensive guide to the best hiking shoes for Patagonia and saved it to **`patagonia_hiking_shoes_recommendations.txt`** in your current directory.

## Quick Summary of Top Recommendations:

**🥾 Best Overall: Salomon Quest 4 GTX** ($230-260)
- Perfect balance of support, waterproofing, and value
- Ideal for multi-day treks like Torres del Paine

**🏔️ Most Technical: Scarpa Zodiac Plus GTX** ($290-320)
- Crampon-compatible for serious mountain terrain

**👟 Best Lightweight: Salomon X Ultra 4 GTX** ($150-180)
- Great for day hikes and lighter loads

**Key Reminders:**
- ⚠️ **Break them in 4-6 weeks before your trip!**
- Get Gore-Tex waterproofing (essential for Patagonia)
- Size up half a size for thick socks
- Don't skimp on footwear - it's your most important gear

The file includes detailed specifications, buying tips, and terrain considerations. You can reference it when shopping for your boots!

Lets look at the messages of the orchestrator. Here you can see the agent decided to use the sub-agent as tool

In [36]:
orchestrator.messages

[{'role': 'user',
  'content': [{'text': 'Can you do a research on patagonia? Also help me plan a 7 day trip.'}]},
 {'role': 'assistant',
  'content': [{'toolUse': {'toolUseId': 'tooluse_YeG8lf5r8wfnpOksqoeXFO',
     'name': 'research_assistant',
     'input': {'query': 'Patagonia region - geography, climate, attractions, best time to visit, and travel information'}}},
   {'toolUse': {'toolUseId': 'tooluse_KmpZUZXogHj85xZNlLvS9U',
     'name': 'trip_planning_assistant',
     'input': {'query': 'Plan a 7 day trip to Patagonia including main attractions, activities, accommodations, and daily itinerary'}}}]},
 {'role': 'user',
  'content': [{'toolResult': {'toolUseId': 'tooluse_YeG8lf5r8wfnpOksqoeXFO',
     'status': 'success',
     'content': [{'text': '# Patagonia Region: Comprehensive Travel Guide\n\n## Geography\n\nPatagonia is a vast region spanning the southern portions of **South America**, divided between:\n\n- **Argentina** (eastern side): Approximately 260,000 sq miles\n- **Chil

In [37]:
customer_query = "Can you help me plan my trip to Patagonia"

response = orchestrator(customer_query)

I'd love to help you plan your Patagonia trip! I actually already created a detailed 7-day itinerary for you earlier in our conversation. Let me know if you'd like me to:

**Option 1: Review the Previous Itinerary**
The itinerary I created covers:
- Days 1-2: El Calafate & Perito Moreno Glacier (Argentina)
- Days 3-4: El Chaltén & Fitz Roy hike (Argentina)
- Days 5-7: Torres del Paine (Chile)
- Budget: $2,500-4,000 USD

**Option 2: Create a NEW Custom Itinerary**
If you'd like something different, please tell me:

1. **How many days?** (Weekend getaway, 1 week, 2 weeks, longer?)

2. **When are you traveling?** (Month/season)

3. **What's your style?**
   - Hardcore trekking adventure
   - Moderate hiking with comfort
   - Relaxed sightseeing
   - Wildlife focused
   - Photography focused

4. **Fitness level?** (Comfortable with long hikes, or prefer easier activities?)

5. **Budget?** (Backpacker, mid-range, or luxury)

6. **Any must-sees?** (Specific glaciers, peaks, wildlife, etc.)



In [35]:
orchestrator.messages

[{'role': 'user',
  'content': [{'text': 'Can you do a research on patagonia? Also help me plan a 7 day trip.'}]},
 {'role': 'assistant',
  'content': [{'toolUse': {'toolUseId': 'tooluse_YeG8lf5r8wfnpOksqoeXFO',
     'name': 'research_assistant',
     'input': {'query': 'Patagonia region - geography, climate, attractions, best time to visit, and travel information'}}},
   {'toolUse': {'toolUseId': 'tooluse_KmpZUZXogHj85xZNlLvS9U',
     'name': 'trip_planning_assistant',
     'input': {'query': 'Plan a 7 day trip to Patagonia including main attractions, activities, accommodations, and daily itinerary'}}}]},
 {'role': 'user',
  'content': [{'toolResult': {'toolUseId': 'tooluse_YeG8lf5r8wfnpOksqoeXFO',
     'status': 'success',
     'content': [{'text': '# Patagonia Region: Comprehensive Travel Guide\n\n## Geography\n\nPatagonia is a vast region spanning the southern portions of **South America**, divided between:\n\n- **Argentina** (eastern side): Approximately 260,000 sq miles\n- **Chil

### Calling multiple agents 

In [38]:
orchestrator.messages = []

In [39]:
query = "Can you do a research on patagonia? Also help me plan a 7 day trip."

orchestrator(query)

I'll help you with both! Let me gather research information about Patagonia and create a 7-day trip plan for you.
Tool #16: research_assistant

Tool #17: trip_planning_assistant
# 7-Day Patagonia Adventure Itinerary

## Overview
This itinerary covers the highlights of Argentine an# Patagonia: Comprehensive Travel and Geographic Information

## Geographic Overview

**Location and Extent:**
Patagonia is a vastd Chilean Patagonia, region spanning focusing the on El southern portions Calafate, El of South America, share Chaltén, and Torres del Paine.d between **Argentina

---

## ** (approximatelyDay 1: Arrival in 90%) El Calafate, and Chile (approximately 10% Argentina**

### Morning/Afternoon
-)**. It extends Arrive at El Calafate International from the Colora Airport (do RiverF (around 40TE°S latitude)
- Check into hotel) south in townward to the Strait of Ma center
- Restgellan an and accd Tierralimatize

### Evening
- St del Fuego,roll along covering approximately ** A1venida del, Lib

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Perfect! I've gathered comprehensive information about Patagonia and created a detailed 7-day itinerary for you.\n\n## Research Summary on Patagonia\n\nPatagonia is a breathtaking region at the southern tip of South America, shared between Argentina and Chile. Here are the key highlights:\n\n**Geography:** Covers over 1 million square kilometers, featuring dramatic landscapes from glaciers and mountains to fjords and steppes.\n\n**Climate:** Cool and windy, with the best travel season being December-March (summer). Expect temperatures of 15-25°C (59-77°F) in summer and strong westerly winds.\n\n**Top Attractions:**\n- Torres del Paine National Park (Chile)\n- Los Glaciares National Park with Perito Moreno Glacier (Argentina)\n- Mount Fitz Roy and El Chaltén\n- Peninsula Valdés for whale watching\n- Ushuaia - the world's southernmost city\n\n**Wildlife:** Whales, penguins, guanacos, pumas, and Andean

Behind the scenes, the orchestrator will:
1. First call the `research_assistant`
2. Then call `trip_planning_assistant`
3. Combine these specialized responses into a cohesive answer that addresses both the queries

### Sequential Agent Communication Pattern


The agent tool can also combine multiple agents together. In this example we will provide output of `research_agent` to `summary_agent` and return the summarized response.

In [40]:
 # define the user query
topic = "generative Ai"
# Create a research agent
research_agent = Agent(
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt=RESEARCH_ASSISTANT_PROMPT,
)
# Create a summarization agent
summary_agent = Agent(
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt="""
    You are a summarization specialist focused on distilling complex information into clear, concise summaries.
    Your primary goal is to take detailed information and extract the key points, main arguments, and critical data.
    You should maintain the accuracy of the original content while making it more digestible.
    Focus on clarity, brevity, and highlighting the most important aspects of the information.
    """,
)

print("Multiple agents created successfully!")
print(f"\n🔍 RESEARCH AGENT working on: {topic}\n") 
try:
    # Agent 1: Invoke research agent
    research_response = research_agent(
        f"Please gather comprehensive information about {topic}."
    )
    research_text = research_response.message['content'][0]["text"]
    print("\n✂️ SUMMARY AGENT distilling the research\n")
    
    # Agent 2: Ask the summary agent to create a concise summary
    summary_response = summary_agent(
        f"Please create a concise summary of this research: {research_text}"
    )
    summary_text = summary_response.message['content'][0]["text"]
    
    print(summary_text)
except Exception as e:
    print(f"Error in research assistant: {str(e)}")

Multiple agents created successfully!

🔍 RESEARCH AGENT working on: generative Ai

# Comprehensive Information About Generative AI

## Definition and Overview

Generative AI refers to artificial intelligence systems that can create new content—including text, images, audio, video, code, and other data types—based on patterns learned from training data. Unlike traditional AI systems that classify or analyze existing data, generative AI produces novel outputs that resemble but don't replicate its training examples.

## Core Technologies

### 1. **Large Language Models (LLMs)**
- Based on transformer architecture introduced in the 2017 paper "Attention Is All You Need" by Vaswani et al.
- Examples: GPT (Generative Pre-trained Transformer) series, BERT, LLaMA, Claude, PaLM
- Trained on massive text corpora to understand and generate human-like text

### 2. **Diffusion Models**
- Used primarily for image generation
- Examples: DALL-E, Midjourney, Stable Diffusion
- Work by learning to rever


## Congrats!

You've learned how to use agents as tools in Strands Agents to create more complex agentic applications